<a href="https://colab.research.google.com/github/rogerracer73/curso-facens-linguagem-phyton/blob/main/TrabalhoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Exercício: Análise Manual do Dataset Disney+

**Objetivo:**

Este exercício tem como objetivo desenvolver suas habilidades em manipulação manual de dados em Python, processando um dataset de shows da Disney+ sem o auxílio de bibliotecas de parsing como `csv` ou `pandas`. Você deverá extrair informações relevantes e gerar um relatório detalhado.

**Dataset:**

O dataset a ser utilizado está disponível no seguinte link:
[https://www.kaggle.com/datasets/eshummalik/disney](https://www.kaggle.com/datasets/eshummalik/disney)

**Instruções:**

1.  **Download do Dataset:** Baixe o arquivo do dataset manualmente a partir do link fornecido.
2.  **Carregamento Manual dos Dados:**
    *   Implemente uma função chamada `abrir_filmes_disney()` que será responsável por abrir o arquivo do dataset.
    *   Você **não** deve utilizar bibliotecas como `csv` ou `pandas` para ler o arquivo. A leitura deve ser feita linha por linha, utilizando as funcionalidades básicas de manipulação de arquivos em Python.
    *   Ignore linhas vazias encontradas no arquivo.
    *   Para cada linha do dataset, identifique se ela representa um filme (`type` coluna) ou uma série.
    *   Crie duas classes em Python: `Filme` e `Serie`. Cada linha do dataset deve ser mapeada para uma instância da classe `Filme` ou `Serie` correspondente, contendo seus respectivos atributos (baseados nas colunas do dataset). Implemente getters e setters para os atributos conforme necessário.
    *   Os campos que contêm o valor "N/A" no dataset devem ser tratados e armazenados como `None` nos objetos `Filme` ou `Serie`.
    *   A função `abrir_filmes_disney()` deve retornar uma lista contendo todos os objetos `Filme` e `Serie` criados a partir do dataset.
    *   Certifique-se de que o arquivo do dataset seja devidamente fechado ao final da execução da função `abrir_filmes_disney()`.
3.  **Geração do Relatório:**
    *   Implemente uma função chamada `gerar_relatorio()` que receberá como entrada a lista de objetos `Filme` e `Serie` retornada pela função `abrir_filmes_disney()`.
    *   Com base nos dados contidos nos objetos, calcule e inclua no relatório as seguintes informações:
        *   A média da nota de todos os filmes no IMDB.
        *   A média da nota de todos os filmes no Metascore.
        *   A média do número de votos de todos os filmes no IMDB.
        *   As 3 línguas mais usadas e as 3 línguas menos usadas no dataset.
        *   Os 3 atores que mais aparecem e os 3 atores que menos aparecem no dataset.
        *   O diretor com mais filmes no dataset.
        *   O diretor com o filme mais popular no IMDB (considerando a maior nota IMDB).
        *   O diretor com o filme mais popular no Metascore (considerando a maior nota Metascore).
        *   O ano em que mais filmes foram lançados.
        *   A pior série segundo a nota IMDB e a pior série segundo a nota Metascore.
        *   Uma lista de filmes que possuem mais de um lançamento (considerados "remakes" ou diferentes versões no dataset, identificados por títulos iguais mas anos de lançamento diferentes).
    *   O relatório de saída deve ser salvo em um arquivo texto chamado `relatorio-disney.txt` no mesmo diretório do script.
4.  **Estrutura do Código:** Organize seu código de forma modular, utilizando as funções `abrir_filmes_disney()` e `gerar_relatorio()` conforme especificado. Você não é obrigado a passar parâmetros para essas funções, mas pode fazê-lo se julgar necessário para uma melhor organização do código.

In [4]:
# ==========================================================
# Exercício Disney+ — Limpeza, Reconstrução e Relatório Final Completo
# ==========================================================

import re
from typing import List, Dict, Optional

# ----------------------------------------------------------
# Função: corrigir linhas corrompidas e aspas partidas
# ----------------------------------------------------------
def corrigir_linhas_corrompidas(linhas: List[str]) -> List[str]:
    """Une linhas quebradas e corrige aspas desequilibradas e truncamentos."""
    linhas_corrigidas = []
    buffer = ""
    aspas_abertas = False

    for raw in linhas:
        l = raw.strip()
        if not l:
            continue
        l = l.replace("...", " ").replace("\t", " ")
        qtd_aspas = l.count('"')

        if aspas_abertas or (qtd_aspas % 2 == 1):
            buffer += " " + l
            aspas_abertas = not aspas_abertas
            if not aspas_abertas:
                linhas_corrigidas.append(buffer.strip())
                buffer = ""
        else:
            if buffer:
                buffer += " " + l
                linhas_corrigidas.append(buffer.strip())
                buffer = ""
            else:
                linhas_corrigidas.append(l.strip())

    if buffer:
        linhas_corrigidas.append(buffer.strip())
    return [re.sub(r",\s*,", ",", l) for l in linhas_corrigidas]


# ----------------------------------------------------------
# Parser CSV manual
# ----------------------------------------------------------
def parse_csv_line_manual(line: str) -> list:
    fields = []
    current = []
    in_quotes = False
    i = 0
    while i < len(line):
        ch = line[i]
        if ch == '"':
            if in_quotes and i + 1 < len(line) and line[i + 1] == '"':
                current.append('"')
                i += 1
            else:
                in_quotes = not in_quotes
        elif ch == ',' and not in_quotes:
            fields.append(''.join(current))
            current = []
        else:
            current.append(ch)
        i += 1
    fields.append(''.join(current))
    return [f.strip() for f in fields]


# ----------------------------------------------------------
# Classes
# ----------------------------------------------------------
class Filme:
    def __init__(self, imdb_id, title, plot, tipo, rated, year,
                 released_at, added_at, runtime, genre, director,
                 writer, actors, language, metascore, imdb_rating, imdb_votes):
        self.imdb_id = imdb_id
        self.title = title
        self.plot = plot
        self.tipo = tipo
        self.rated = rated
        self.year = year
        self.released_at = released_at
        self.added_at = added_at
        self.runtime = runtime
        self.genre = genre
        self.director = director
        self.writer = writer
        self.actors = actors
        self.language = language
        self.metascore = metascore
        self.imdb_rating = imdb_rating
        self.imdb_votes = imdb_votes


class Serie(Filme):
    pass


# ----------------------------------------------------------
# Funções de limpeza e conversão
# ----------------------------------------------------------
def to_none_if_na(s):
    if not s or s.upper() == "N/A":
        return None
    return s

def parse_float(num):
    if not num:
        return None
    num = num.replace(",", "").strip()
    try:
        return float(num)
    except:
        return None

def parse_int(num):
    if not num:
        return None
    num = num.replace(",", "").strip()
    try:
        return int(float(num))
    except:
        return None

def padronizar(s):
    if s is None:
        return None
    s = re.sub(r"[.,]", "", s.strip().upper())
    s = re.sub(r"\s+", " ", s)
    return s


# ----------------------------------------------------------
# Função principal — Leitura e reconstrução
# ----------------------------------------------------------
def abrir_filmes_disney():
    caminho = input("Digite o nome do arquivo Disney+ a ser processado (padrão: disney_plus_shows.txt): ").strip()
    if not caminho:
        caminho = "disney_plus_shows.txt"

    with open(caminho, "r", encoding="utf-8") as f:
        linhas = [l.rstrip("\n") for l in f if l.strip()]

    linhas = corrigir_linhas_corrompidas(linhas)
    header = linhas[0]
    data = linhas[1:]

    objetos = []

    for linha in data:
        campos = parse_csv_line_manual(linha)
        if len(campos) < 17:
            campos += [""] * (17 - len(campos))
        metascore, imdb_rating, imdb_votes = campos[-3:]
        principais = campos[:-3]
        if len(principais) < 14:
            principais += [""] * (14 - len(principais))
        (imdb_id, title, plot, tipo, rated, year,
         released_at, added_at, runtime, genre,
         director, writer, actors, language) = principais[:14]

        tipo = (tipo or "").strip().lower()
        if tipo not in ("movie", "series") or not title.strip():
            continue

        obj_cls = Filme if tipo == "movie" else Serie
        obj = obj_cls(
            padronizar(to_none_if_na(imdb_id)),
            padronizar(to_none_if_na(title)),
            padronizar(to_none_if_na(plot)),
            padronizar(to_none_if_na(tipo)),
            padronizar(to_none_if_na(rated)),
            padronizar(to_none_if_na(year)),
            padronizar(to_none_if_na(released_at)),
            padronizar(to_none_if_na(added_at)),
            padronizar(to_none_if_na(runtime)),
            padronizar(to_none_if_na(genre)),
            padronizar(to_none_if_na(director)),
            padronizar(to_none_if_na(writer)),
            padronizar(to_none_if_na(actors)),
            padronizar(to_none_if_na(language)),
            parse_float(metascore),
            parse_float(imdb_rating),
            parse_int(imdb_votes),
        )
        objetos.append(obj)

    print(f"\n✅ Arquivo '{caminho}' processado com sucesso.")
    print(f"Registros válidos: {len(objetos)}\n")
    return objetos


# ----------------------------------------------------------
# Funções auxiliares para relatório
# ----------------------------------------------------------
def split_tokens(raw: str) -> list:
    if not raw:
        return []
    parts = re.split(r"[;,/]+", raw)
    return [p.strip().upper() for p in parts if p.strip()]

def media(xs):
    vals = [v for v in xs if isinstance(v, (int, float))]
    return sum(vals)/len(vals) if vals else 0.0


# ----------------------------------------------------------
# Geração do relatório completo
# ----------------------------------------------------------
def gerar_relatorio(objetos):
    filmes = [o for o in objetos if o.tipo == "MOVIE"]
    series = [o for o in objetos if o.tipo == "SERIES"]

    # Médias
    media_imdb_filmes = media([f.imdb_rating for f in filmes])
    media_meta_filmes = media([f.metascore for f in filmes])
    media_votes_filmes = media([f.imdb_votes for f in filmes])
    media_imdb_series = media([s.imdb_rating for s in series])
    media_meta_series = media([s.metascore for s in series])
    media_votes_series = media([s.imdb_votes for s in series])

    # Línguas
    lang_count = {}
    for o in objetos:
        for l in split_tokens(o.language):
            lang_count[l] = lang_count.get(l, 0) + 1
    top3_lang = sorted(lang_count.items(), key=lambda x: -x[1])[:3]
    bot3_lang = sorted(lang_count.items(), key=lambda x: x[1])[:3]

    # Atores
    act_count = {}
    for o in objetos:
        for a in split_tokens(o.actors):
            act_count[a] = act_count.get(a, 0) + 1
    top3_act = sorted(act_count.items(), key=lambda x: -x[1])[:3]
    bot3_act = sorted(act_count.items(), key=lambda x: x[1])[:3]

    # Diretores
    dir_count = {}
    for f in filmes:
        for d in split_tokens(f.director):
            dir_count[d] = dir_count.get(d, 0) + 1
    diretor_mais = max(dir_count.items(), key=lambda x: x[1])[0] if dir_count else "N/A"

    # Filme e série mais populares
    filme_top_imdb = max(filmes, key=lambda f: f.imdb_rating or 0, default=None)
    filme_top_meta = max(filmes, key=lambda f: f.metascore or 0, default=None)
    serie_top_imdb = max(series, key=lambda s: s.imdb_rating or 0, default=None)
    serie_top_meta = max(series, key=lambda s: s.metascore or 0, default=None)

    diretor_top_imdb = split_tokens(filme_top_imdb.director)[0] if filme_top_imdb else "N/A"
    diretor_top_meta = split_tokens(filme_top_meta.director)[0] if filme_top_meta else "N/A"

    # Ano com mais filmes
    anos = {}
    for f in filmes:
        if f.year:
            anos[f.year] = anos.get(f.year, 0) + 1
    ano_top = max(anos.items(), key=lambda x: x[1])[0] if anos else "N/A"

    # Piores séries
    serie_worst_imdb = min(series, key=lambda s: s.imdb_rating or 10, default=None)
    serie_worst_meta = min(series, key=lambda s: s.metascore or 100, default=None)

    # Remakes
    remakes = {}
    for f in filmes:
        if f.title and f.year:
            remakes.setdefault(f.title, set()).add(f.year)
    remakes = {t: anos for t, anos in remakes.items() if len(anos) > 1}

    # Montagem do relatório
    linhas = []
    linhas.append("===== RELATÓRIO DISNEY+ =====")
    linhas.append(f"Total de Filmes: {len(filmes)}")
    linhas.append(f"Total de Séries: {len(series)}")
    linhas.append("")
    linhas.append(f"Média IMDb (filmes): {media_imdb_filmes:.2f}")
    linhas.append(f"Média Metascore (filmes): {media_meta_filmes:.2f}")
    linhas.append(f"Média Votos IMDb (filmes): {media_votes_filmes:.2f}")
    linhas.append("")
    linhas.append(f"Média IMDb (séries): {media_imdb_series:.2f}")
    linhas.append(f"Média Metascore (séries): {media_meta_series:.2f}")
    linhas.append(f"Média Votos IMDb (séries): {media_votes_series:.2f}")
    linhas.append("")
    linhas.append(f"3 Línguas Mais Usadas: {top3_lang}")
    linhas.append(f"3 Línguas Menos Usadas: {bot3_lang}")
    linhas.append("")
    linhas.append(f"3 Atores Mais Frequentes: {top3_act}")
    linhas.append(f"3 Atores Menos Frequentes: {bot3_act}")
    linhas.append("")
    linhas.append(f"Diretor com mais filmes: {diretor_mais}")
    linhas.append(f"Diretor com filme mais popular IMDb: {diretor_top_imdb} ({filme_top_imdb.title if filme_top_imdb else 'N/A'})")
    linhas.append(f"Diretor com filme mais popular Metascore: {diretor_top_meta} ({filme_top_meta.title if filme_top_meta else 'N/A'})")
    linhas.append("")
    linhas.append(f"Melhor série IMDb: {(serie_top_imdb.title if serie_top_imdb else 'N/A')}")
    linhas.append(f"Melhor série Metascore: {(serie_top_meta.title if serie_top_meta else 'N/A')}")
    linhas.append("")
    linhas.append(f"Pior série IMDb: {(serie_worst_imdb.title if serie_worst_imdb else 'N/A')}")
    linhas.append(f"Pior série Metascore: {(serie_worst_meta.title if serie_worst_meta else 'N/A')}")
    linhas.append("")
    linhas.append(f"Ano com mais filmes: {ano_top}")
    linhas.append("")
    linhas.append("Filmes com múltiplos lançamentos:")
    if remakes:
        for t, anos in remakes.items():
            linhas.append(f" - {t}: {', '.join(sorted(list(anos)))}")
    else:
        linhas.append(" - Nenhum encontrado.")

    texto = "\n".join(linhas)
    with open("relatorio-disney.txt", "w", encoding="utf-8") as f:
        f.write(texto)

    print(texto)
    print("\n✅ Relatório salvo em 'relatorio-disney.txt'.")


# ----------------------------------------------------------
# Execução
# ----------------------------------------------------------
objetos = abrir_filmes_disney()
gerar_relatorio(objetos)


Digite o nome do arquivo Disney+ a ser processado (padrão: disney_plus_shows.txt): 

✅ Arquivo 'disney_plus_shows.txt' processado com sucesso.
Registros válidos: 871

===== RELATÓRIO DISNEY+ =====
Total de Filmes: 680
Total de Séries: 191

Média IMDb (filmes): 6.58
Média Metascore (filmes): 62.06
Média Votos IMDb (filmes): 79102.93

Média IMDb (séries): 6.91
Média Metascore (séries): 0.00
Média Votos IMDb (séries): 8115.14

3 Línguas Mais Usadas: [('ENGLISH', 729), ('ENGLISH FRENCH', 17), ('ENGLISH SPANISH', 16)]
3 Línguas Menos Usadas: [('ENGLISH CANTONESE FRENCH GERMAN HINDI TURKISH', 1), ('NORWEGIAN ENGLISH', 1), ('CHINESE ENGLISH', 1)]

3 Atores Mais Frequentes: [('WINSTON HIBLER', 10), ('CLARENCE NASH', 7), ('DESSIE FLYNN JAMES MACDONALD CLARENCE NASH', 6)]
3 Atores Menos Frequentes: [('HEATH LEDGER JULIA STILES JOSEPH GORDON-LEVITT LARISA OLEYNIK', 1), ('JOSH BRENER MICHAELA DIETZ BERT DAVIS ABIGAIL ZOE LEWIS', 1), ('GLENN CLOSE JEFF DANIELS JOELY RICHARDSON JOAN PLOWRIGHT', 1)]
